In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# CrackSegDiff: Simplified Training & Inference
Automated setup, data preparation (First 500 Test / 2000 Train), training, and testing.

In [2]:
# 1. Setup Environment & Weights
!nvidia-smi
import os
if not os.path.exists('CrackSegDiff'):
    !git clone https://github.com/Ludwig-H/CrackSegDiff.git
%cd CrackSegDiff
!git pull

!sed -i 's/beta_end = scale [*] 0.02/beta_end = scale * 0.02\n        if beta_end > 0.999: beta_end = 0.999/' CrackSegDiff/guided_diffusion/gaussian_diffusion.py

# Downgrade PyTorch to stable 2.4.0 for guaranteed Mamba compatibility
print("Installing PyTorch 2.4.0 compatible with Mamba wheels...")
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

!pip install -r requirement.txt

# Force install pre-built wheels for Mamba-SSM and Causal-Conv1d to avoid compilation errors and ensure GPU speed
print("Installing Optimized Mamba Kernels...")
import torch
cuda_version = torch.version.cuda.replace('.', '')
torch_version = torch.__version__.split('+')[0].replace('.', '')
# Assuming standard Colab PyTorch 2.x and CUDA 11.8 or 12.x
# We use the releases from Dao-AILab which are reliable
!pip install ninja  # Speeds up compilation
!pip install causal-conv1d>=1.0.0 --no-build-isolation -v
!pip install mamba-ssm>=1.0.1 --no-build-isolation -v

# If the above standard install fails (it tries to build), we could try finding wheels:
# (But usually --no-build-isolation helps or just standard pip works if env is clean)

# Verify installation
try:
    import mamba_ssm
    print("\nSUCCESS: Mamba SSM installed successfully! GPU acceleration enabled (x10 speed).")
except ImportError:
    print("\nWARNING: Mamba SSM installation failed.")
    print("Fallback to Pure Python active (SLOWER).")

# Download Pretrained Weights
!mkdir -p pretrained_weights
!gdown 1JYqMxM5dbCLZ-WGPKtIofYJhj0VPuy3l -O pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth

# Patch Hardcoded Paths
target_file = 'CrackSegDiff/guided_diffusion/unet.py'
new_path = os.path.abspath('pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth')
if os.path.exists(target_file):
    with open(target_file, 'r') as f: content = f.read()
    content = content.replace('/home/dell/jlc/segdiff/pre_trained_weights/vssm_base_0229_ckpt_epoch_237.pth', new_path)
    with open(target_file, 'w') as f: f.write(content)
    print("Path patched successfully.")

Thu Jan  8 01:57:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Installing Optimized Mamba Kernels...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.9 MB/s eta 0:00:00
  Running command Preparing metadata (pyproject.toml)


  torch.__version__  = 2.4.0+cu121


  running dist_info
  creating /tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info
  writing /tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info/requires.txt
  writing top-level names to /tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-3vo84vk5/causal_conv1d.egg-info/SOURCES.txt'
  reading manifest template 'MANIFEST.in'

  adding license file 'LICENSE'
  adding license file 'AUTHORS'
  writing manifest file

In [3]:
# 2. Prepare Data (First 500 Test / 2000 Train)
!rm -rf data && mkdir -p data
%cd data
!gdown 1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK -O data.zip
!unzip -q -o data.zip
%cd ..

import os, glob, shutil
print("Organizing Data...")

# Find folders
try:
    src_img = glob.glob('data/**/img/fused', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]
except IndexError:
    # Fallback if 'fused' not found, try 'intensity'
    print("Fused folder not found, checking intensity...")
    src_img = glob.glob('data/**/img/intensity', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]

print(f"Images source: {src_img}")
print(f"Labels source: {src_lbl}")

# Get sorted file lists
img_files = sorted(glob.glob(os.path.join(src_img, '*')))
lbl_files = sorted(glob.glob(os.path.join(src_lbl, '*.bmp')))

# Split: First 500 Test, Rest Train
test_pairs = list(zip(img_files[:500], lbl_files[:500]))
train_pairs = list(zip(img_files[500:], lbl_files[500:]))

print(f"Test Set: {len(test_pairs)} (First 500)")
print(f"Train Set: {len(train_pairs)} (Rest)")

# Copy to formatted directories
train_dir = os.path.abspath('data/Train')
test_dir = os.path.abspath('data/Test')

for pairs, dest in [(train_pairs, train_dir), (test_pairs, test_dir)]:
    os.makedirs(os.path.join(dest, '5d'), exist_ok=True)
    os.makedirs(os.path.join(dest, 'mask'), exist_ok=True)
    for img, mask in pairs:
        shutil.copy(img, os.path.join(dest, '5d'))
        shutil.copy(mask, os.path.join(dest, 'mask'))
print("Data preparation complete.")

/content/CrackSegDiff/data
Downloading...
From (original): https://drive.google.com/uc?id=1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK
From (redirected): https://drive.google.com/uc?id=1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK&confirm=t&uuid=6cc6f0ea-208a-4553-9f92-1c7dee31da6f
To: /content/CrackSegDiff/data/data.zip
100% 1.15G/1.15G [00:41<00:00, 27.6MB/s]
/content/CrackSegDiff
Organizing Data...
Images source: data/data/img/fused
Labels source: data/data/lbs
Test Set: 500 (First 500)
Train Set: 2000 (Rest)
Data preparation complete.


In [ ]:
# 3. Train Model
import os
data_dir = os.path.abspath('data/Train')
out_dir = os.path.abspath('results/train_output')
os.makedirs(out_dir, exist_ok=True)

!python CrackSegDiff/segmentation_train.py --data_dir {data_dir} --out_dir {out_dir} --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --lr 5e-5 --batch_size 8 --save_interval 5000 --lr_anneal_steps 40000

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
Step 3170: Loss = 1.3306
Step 3180: Loss = 1.7139
Step 3190: Loss = 1.3972
Step 3200: Loss = 1.6536
---------------------------
| grad_norm    | 7.95     |
| loss         | 0.00323  |
| loss_cal     | 0.152    |
| loss_cal_q0  | 0.152    |
| loss_cal_q1  | 0.15     |
| loss_cal_q2  | 0.152    |
| loss_cal_q3  | 0.154    |
| loss_diff    | 0.00283  |
| loss_diff_q0 | 0.00805  |
| loss_diff_q1 | 0.00226  |
| loss_diff_q2 | 0.000579 |
| loss_diff_q3 | 0.000344 |
| loss_q0      | 0.00963  |
| loss_q1      | 0.00227  |
| loss_q2      | 0.000584 |
| loss_q3      | 0.000348 |
| param_norm   | 2.06e+03 |
| samples      | 2.56e+04 |
| step         | 3.2e+03  |
| vb           | 0.000408 |
| vb_q0        | 0.00157  |
| vb_q1        | 1.67e-05 |
| vb_q2        | 5.21e-06 |
| vb_q3        | 4.42e-06 |
---------------------------
Step 3210: Loss = 1.3719
Step 3220: Loss = 1.4183
Step 3230: Loss = 1.3972
Step 3240: Loss = 2

In [ ]:
# 4. Inference
import glob, os
# Use latest trained model
models = sorted(glob.glob('results/train_output/*.pt'))
model_path = models[-1] if models else "pretrained_weights/savedmodel100000.pt"
test_dir = os.path.abspath('data/Test')
print(f"Using model: {model_path}")

for modality in ['intensity', 'range', 'fused']: # , 'filtered', 'all'
    out_path = f"results/test_output_{modality}"
    os.makedirs(out_path, exist_ok=True)
    print(f"Testing {modality}...")
    !python CrackSegDiff/segmentation_sample.py --data_dir {test_dir} --out_dir {out_path} --model_path {model_path} --modality {modality} --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1
    print(f"Done {modality}")

In [ ]:
# 5. Zip Results
!zip -r results.zip results

In [ ]:
# 6. Robustness Benchmark (Noise Experiments)
# This section generates noisy datasets (Speckle/Gaussian), runs inference, and saves to Drive
# to match the expected input for 'find_frangi_fusion_avignon_colab.py'.

import numpy as np
import os
import glob
import shutil
from skimage import io
from tqdm import tqdm
from PIL import Image

# --- 6.1 Configuration & Noise Logic (Exact Match) ---
NOISE_BASE_SEED = 133
DRIVE_ROOT = "/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise"
TEMP_DATA_ROOT = os.path.abspath("temp_noise_exp")
MODEL_PATH = sorted(glob.glob('results/train_output/*.pt'))[-1] if glob.glob('results/train_output/*.pt') else "pretrained_weights/savedmodel100000.pt"

speckle_vars = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
range_sigmas = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]

def _normalize01(x: np.ndarray):
    x = np.asarray(x).astype(np.float32)
    mn = float(np.min(x))
    mx = float(np.max(x))
    if (mx - mn) < 1e-12:
        return np.zeros_like(x, dtype=np.float32), mn, mx
    return (x - mn) / (mx - mn), mn, mx

def _denormalize01(x01: np.ndarray, mn: float, mx: float):
    return x01 * (mx - mn) + mn

def add_speckle_intensity(x: np.ndarray, var: float, rng: np.random.Generator):
    if var <= 0:
        return np.asarray(x).astype(np.float32)
    x01, mn, mx = _normalize01(x)
    n = rng.normal(0.0, np.sqrt(var), size=x01.shape).astype(np.float32)
    y01 = x01 + x01 * n
    y01 = np.clip(y01, 0.0, 1.0)
    return _denormalize01(y01, mn, mx).astype(np.float32)

def add_gaussian_range(x: np.ndarray, sigma: float, rng: np.random.Generator):
    if sigma <= 0:
        return np.asarray(x).astype(np.float32)
    x01, mn, mx = _normalize01(x)
    y01 = x01 + rng.normal(0.0, sigma, size=x01.shape).astype(np.float32)
    y01 = np.clip(y01, 0.0, 1.0)
    return _denormalize01(y01, mn, mx).astype(np.float32)

def _noise_tag(x: float, ndigits: int = 4):
    return f"{x:.{ndigits}f}".replace(".", "p")

# --- 6.2 Processing Loop ---
experiments = [
    ("speckle_intensity", speckle_vars),
    ("gauss_range", range_sigmas),
    ("both", range_sigmas)
]

print(f"Starting Robustness Benchmark. Saving to: {DRIVE_ROOT}")
print(f"Using Model: {MODEL_PATH}")

clean_test_dir = os.path.abspath('data/Test')
src_imgs = sorted(glob.glob(os.path.join(clean_test_dir, '5d', '*')))
src_masks = sorted(glob.glob(os.path.join(clean_test_dir, 'mask', '*')))

# Ensure we only take first 500 if more exist (though Setup cell handled this)
src_imgs = src_imgs[:500]
src_masks = src_masks[:500]

for exp_name, levels in experiments:
    print(f"\n=== Experiment: {exp_name} ===")

    for level_id, lvl in enumerate(levels):
        lvl = float(lvl)
        tag = _noise_tag(lvl)

        # Define Output Path
        final_dest_dir = os.path.join(DRIVE_ROOT, exp_name, tag, "test_output_fused")
        if os.path.exists(final_dest_dir) and len(os.listdir(final_dest_dir)) >= 500:
            print(f"Skipping {exp_name} - {tag} (Already exists)")
            continue

        print(f"Processing Level {level_id}: {lvl} (Tag: {tag})")

        # 1. Prepare Noisy Data
        current_data_dir = os.path.join(TEMP_DATA_ROOT, exp_name, tag)
        img_dir = os.path.join(current_data_dir, '5d')
        msk_dir = os.path.join(current_data_dir, 'mask')
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(msk_dir, exist_ok=True)

        # Generate Images
        for idx, (img_path, msk_path) in enumerate(zip(src_imgs, src_masks)):
            # Deterministic Seeds (Critical for alignment with analysis script)
            rng_I = np.random.default_rng(NOISE_BASE_SEED + 100000 * idx + 97 * level_id + 1)
            rng_R = np.random.default_rng(NOISE_BASE_SEED + 100000 * idx + 97 * level_id + 2)

            # Load Fused (Intensity=Ch0, Range=Ch1 usually, or just Intensity if single)
            # But CrackSegDiff data preparation copies 'fused' images which are usually 3-channel or 1-channel.
            # We need to assume the input '5d' contains the data we want to noise.
            # NOTE: The notebook prep copies from 'data/**/img/fused'.
            # If fused is RGB, typically Ch0=Int, Ch1=Range. Let's handle generic load.

            img = io.imread(img_path)

            # Setup Noise Params
            s_var, r_sig = 0.0, 0.0
            if exp_name == "speckle_intensity":
                s_var = lvl
            elif exp_name == "gauss_range":
                r_sig = lvl
            elif exp_name == "both":
                s_var = lvl
                r_sig = lvl

            # Apply Noise
            # We assume the image is grayscale or 3-channel where R=Intensity, G=Range (standard FIND fusion)
            # If grayscale, we just apply Speckle (if Int) or Gauss (if Range).
            # But 'Fused' implies we have both info.
            # CrackSegDiff assumes 'fused' input.

            noisy_img = img.copy().astype(np.float32)

            if img.ndim == 3:
                # Channel 0: Intensity, Channel 1: Range (Common convention)
                if s_var > 0: noisy_img[..., 0] = add_speckle_intensity(noisy_img[..., 0], s_var, rng_I)
                if r_sig > 0: noisy_img[..., 1] = add_gaussian_range(noisy_img[..., 1], r_sig, rng_R)
                # Apply range noise to filtered channel (Ch2) if exists/requested, similar to analysis script
                if img.shape[2] > 2 and r_sig > 0:
                     noisy_img[..., 2] = add_gaussian_range(noisy_img[..., 2], r_sig, rng_R)
            else:
                # Single channel - Ambiguous. Default to Speckle if Int exp, Gauss if Range exp.
                if s_var > 0: noisy_img = add_speckle_intensity(noisy_img, s_var, rng_I)
                elif r_sig > 0: noisy_img = add_gaussian_range(noisy_img, r_sig, rng_R)

            # Save
            out_name = os.path.basename(img_path)
            # Ensure uint8
            noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
            io.imsave(os.path.join(img_dir, out_name), noisy_img, check_contrast=False)

            # Copy Mask
            shutil.copy(msk_path, os.path.join(msk_dir, os.path.basename(msk_path)))

        # 2. Run Inference
        temp_out_dir = os.path.join(current_data_dir, "output")
        os.makedirs(temp_out_dir, exist_ok=True)

        print(f"Running Inference for {exp_name} - {tag}...")
        !python CrackSegDiff/segmentation_sample.py --data_dir {current_data_dir} --out_dir {temp_out_dir} --model_path {MODEL_PATH} --modality fused --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1

        # 3. Save to Drive
        print(f"Saving to Drive: {final_dest_dir}")
        os.makedirs(final_dest_dir, exist_ok=True)

        # Copy and ensure naming convention imXXXXX_output_ens.png
        # segmentation_sample outputs are typically like 'im00001_output_ens.png' if script unmodified,
        # or 'output_ens.png' depending on batch.
        # We'll just glob pngs.
        for png in glob.glob(os.path.join(temp_out_dir, "*.png")):
            shutil.copy(png, final_dest_dir)

        # Cleanup Temp for this level to save space
        shutil.rmtree(current_data_dir)

print("Robustness Benchmark Complete.")